# KV-cache attention kernels

This notebook runs the MHA/GQA/MQA forward and backward checks, grouped decode, split-KV, Paged Attention, and absorbed MLA checks from the companion Triton implementation. Colab should connect to a GPU runtime automatically; if it does not, select **Runtime → Change runtime type → GPU**.

In [ ]:
import subprocess
import sys
from pathlib import Path

import torch

if not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU is attached. Select Runtime → Change runtime type → GPU.")

print("PyTorch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0))
print("bf16 supported:", torch.cuda.is_bf16_supported())

try:
    import triton
except ModuleNotFoundError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "triton"], check=True)
    import triton

print("Triton:", triton.__version__)

In [ ]:
repo = Path("/content/G-U-N.github.io")
if repo.exists():
    subprocess.run(["git", "-C", str(repo), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "https://github.com/G-U-N/G-U-N.github.io.git", str(repo)], check=True)

print("Repository:", repo)

## Run all fp16 correctness checks

This is the portable path for Colab GPUs, including T4. A successful run ends with `All KV-cache attention checks passed.`

In [ ]:
subprocess.run(
    [sys.executable, str(repo / "blogs/code/kv_cache_attention.py"), "--dtype", "fp16"],
    cwd=repo,
    check=True,
)

## Optionally run bf16

The cell skips this check on GPUs without bf16 support.

In [ ]:
if torch.cuda.is_bf16_supported():
    subprocess.run(
        [sys.executable, str(repo / "blogs/code/kv_cache_attention.py"), "--dtype", "bf16"],
        cwd=repo,
        check=True,
    )
else:
    print("Skipping bf16: this GPU does not support it.")